In [1]:
"""
Patrones Binarios Locales (LBP) para el Análisis de Texturas

Objetivo
Presentar los Patrones Binarios Locales (LBP) como un descriptor de textura clásico para el análisis de imágenes.

Importancia
En el notebook anterior, exploramé descriptores de textura simples como la intensidad media, la desviación estándar, la entropía y la varianza local. Estas características son útiles, pero solo capturan información estadística general.

LBP proporciona una forma más estructurada de describir la textura local comparando cada píxel con su vecindario.
"""

'\nPatrones Binarios Locales (LBP) para el Análisis de Texturas\n\nObjetivo\nPresentar los Patrones Binarios Locales (LBP) como un descriptor de textura clásico para el análisis de imágenes.\n\nImportancia\nEn el notebook anterior, exploramé descriptores de textura simples como la intensidad media, la desviación estándar, la entropía y la varianza local. Estas características son útiles, pero solo capturan información estadística general.\n\nLBP proporciona una forma más estructurada de describir la textura local comparando cada píxel con su vecindario.\n'

In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from scipy.ndimage import uniform_filter
from skimage import data
from skimage.feature import local_binary_pattern
from skimage.measure import shannon_entropy

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.decomposition import PCA

In [3]:
np.random.seed(42)
def show_images(images, titles, cmap='gray', figsize=(14, 8), save_path=None):
    n = len(images)
    cols = min(3, n)
    rows = int(np.ceil(n / cols))
    
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).reshape(-1)
    
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img, cmap=cmap)
        ax.set_title(title)
        ax.axis('off')
    
    for ax in axes[len(images):]:
        ax.axis('off')
    
    plt.tight_layout()

    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()


def extract_random_patches(image, patch_size=32, n_patches=40):
    patches = []
    h, w = image.shape
    
    for _ in range(n_patches):
        i = np.random.randint(0, h - patch_size)
        j = np.random.randint(0, w - patch_size)
        patch = image[i:i+patch_size, j:j+patch_size]
        patches.append(patch)
    
    return patches


def compute_lbp(image, P=8, R=1, method='uniform'):
    return local_binary_pattern(image, P=P, R=R, method=method)


def lbp_histogram(image, P=8, R=1, method='uniform', normalize=True):
    lbp = compute_lbp(image, P=P, R=R, method=method)
    
    if method == 'uniform':
        n_bins = P + 2
    else:
        n_bins = int(lbp.max() + 1)
    
    hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, n_bins + 1), density=normalize)
    return hist


def local_variance(image, size=7):
    image = image.astype(np.float32)
    mean = uniform_filter(image, size=size)
    mean_sq = uniform_filter(image**2, size=size)
    return mean_sq - mean**2
